<a href="https://colab.research.google.com/github/SarwatMajeed24/PIAIC/blob/main/LangChain_RAG_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project # 2**
# LangChain RAG Project:
In this Project, we will create a simple LangChain RAG Colab Notebook that uses the Google Gemini Flash model to answer user questions from the document provided. This example below is provided to help you get started assumes you have access to the Gemini API, Pinecone and a basic Python environment. However, you are required to develop and submit your project using Google Colab.

In [ ]:
%pip install -qU langchain-pinecone langchain-google-genai

In [ ]:
from google.colab import userdata


from pinecone import Pinecone, ServerlessSpec

pinecone_api_key =userdata.get("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
import time

index_name = "langchain"  # change if desired


pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

os.environ["GOOGLE_API_KEY"] =userdata.get("GOOGLE_API_KEY")

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")


In [ ]:
vector = embeddings.embed_query("Alishba jabbar")
vector[:5]

[0.03790722414851189,
 -0.04218188673257828,
 -0.022738687694072723,
 -0.004097426775842905,
 0.1065434068441391]

In [ ]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [ ]:
from langchain_core.documents import Document

# Create place info about special places in Pakistan
place_info_1 = Document(
    page_content="The Hunza Valley is known for its breathtaking landscapes, lush greenery, and snow-capped mountains.",
    metadata={"source": "travel_blog"},
)


In [ ]:
place_info_1

Document(metadata={'source': 'travel_blog'}, page_content='The Hunza Valley is known for its breathtaking landscapes, lush greenery, and snow-capped mountains.')

In [ ]:
#Data Save

from uuid import uuid4
from langchain_core.documents import Document

# Create place info about special places in Pakistan
place_info_1 = Document(
    page_content="The Hunza Valley is known for its breathtaking landscapes, lush greenery, and snow-capped mountains.",
    metadata={"source": "travel_blog"},
)

place_info_2 = Document(
    page_content="The Badshahi Mosque in Lahore is a marvel of Mughal architecture, attracting tourists from all over the world.",
    metadata={"source": "historical_site"},
)

place_info_3 = Document(
    page_content="The Karakoram Highway is the highest paved international road in the world, connecting Pakistan to China.",
    metadata={"source": "travel_guide"},
)

place_info_4 = Document(
    page_content="Mohenjo-Daro is an ancient city of the Indus Valley Civilization and a UNESCO World Heritage site.",
    metadata={"source": "history_book"},
)

place_info_5 = Document(
    page_content="The Swat Valley is often called the 'Switzerland of Pakistan' due to its stunning natural beauty.",
    metadata={"source": "travel_blog"},
)

place_info_6 = Document(
    page_content="The Faisal Mosque in Islamabad is one of the largest mosques in the world, with a unique contemporary design.",
    metadata={"source": "website"},
)

place_info_7 = Document(
    page_content="The Deosai Plains, also known as the Land of Giants, offer vast, serene landscapes and rare wildlife.",
    metadata={"source": "travel_guide"},
)

place_info_8 = Document(
    page_content="The Pakistan Monument in Islamabad symbolizes the unity and heritage of Pakistan, with its petal-shaped structure.",
    metadata={"source": "website"},
)

place_info_9 = Document(
    page_content="The Ranikot Fort, also known as the 'Great Wall of Sindh,' is one of the largest forts in the world.",
    metadata={"source": "historical_site"},
)

place_info_10 = Document(
    page_content="The Makran Coastal Highway offers stunning views of the Arabian Sea and unique rock formations like the Sphinx of Balochistan.",
    metadata={"source": "travel_blog"},
)

# List of place info
place_infos = [
    place_info_1,
    place_info_2,
    place_info_3,
    place_info_4,
    place_info_5,
    place_info_6,
    place_info_7,
    place_info_8,
    place_info_9,
    place_info_10,
]




In [ ]:
len(place_infos)

10

In [ ]:
from uuid import uuid4
uuid4()

UUID('cd97dfa0-4036-42c4-83b0-dcc2552b5b3f')

In [ ]:
# Generate unique IDs for each place info
uuids = [str(uuid4()) for _ in range(len(place_infos))]

# Add place info to the vector store
vector_store.add_documents(documents=place_infos, ids=uuids)

['e6a25227-b5ff-4083-99a6-026883edc6fd',
 'de895d90-7636-4daa-a28c-dff5452fa7a7',
 'c7ef2931-89ec-48c2-aaee-df79a0c00e3f',
 'e0611ba3-fc80-499d-b263-3d25a5b7e42f',
 '88f0a980-f4f0-43aa-8582-9848dc98edd2',
 '25ab227e-830f-48d8-bd47-2f8bd04579e2',
 '13c05422-9863-432c-9e37-b8bcf02e49ca',
 '0d286c23-a44d-4dbb-b9be-ee52e376abf4',
 'a34dc27e-f850-4d4b-8fd4-d5b8507ec000',
 '04f00cd6-0e3b-4774-81cb-17414f88a0f2']

In [ ]:
# Search for similar places in Pakistan based on the query
results = vector_store.similarity_search(
    "Which are the most beautiful natural landscapes in Pakistan?",
    k=2,  # Number of results to return
    filter={"source": "travel_blog"},  # Filter to limit results to travel blogs
)

# Print the results
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")


* The Swat Valley is often called the 'Switzerland of Pakistan' due to its stunning natural beauty. [{'source': 'travel_blog'}]
* The Hunza Valley is known for its breathtaking landscapes, lush greenery, and snow-capped mountains. [{'source': 'travel_blog'}]


In [ ]:
# Search for similar places in Pakistan with similarity scores
results = vector_store.similarity_search_with_score(
    "Tell me about famous historical sites in Pakistan.",  # Updated query
    k=1,  # Number of results to return
    filter={"source": "historical_site"},  # Filter to limit results to historical sites
)

# Print the results with similarity scores
for res, score in results:
    print(f"* [SIM={score:.3f}] {res.page_content} [{res.metadata}]")


* [SIM=0.647] The Badshahi Mosque in Lahore is a marvel of Mughal architecture, attracting tourists from all over the world. [{'source': 'historical_site'}]


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

In [ ]:
def answer_to_user(query: str):
    # Perform vector search with filter for relevant source (e.g., travel blog, historical sites)
    vector_results = vector_store.similarity_search(query, k=2, filter={"source": "travel_blog"})  # or another source
    print(len(vector_results ))

    # Prepare the results for LLM to generate an answer
    references = "\n".join([f"* {res.page_content} [{res.metadata}]" for res in vector_results])

    # Final answer by invoking the LLM
    final_answer = llm.invoke(f"ANSWER THIS USER QUERY: {query} Here are some references to answer:\n{references}")

    return final_answer


In [ ]:
answer=answer_to_user("Which are the most beautiful natural landscapes in Pakistan?")

2


In [ ]:
answer.content

'Based on the provided information, two of the most beautiful natural landscapes in Pakistan are:\n\n* **The Swat Valley:** Often referred to as the "Switzerland of Pakistan," it\'s renowned for its stunning natural beauty.\n\n* **The Hunza Valley:**  Famous for its breathtaking landscapes, lush greenery, and snow-capped mountains.\n'